<a href="https://colab.research.google.com/github/barbosa1991/Single_imune/blob/main/project_single_immune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Temos uma matriz contendo uma grande amostra de dados obtidos a partir de sequenciamento de scRNA.

*O primeiro passo do script seria selecionar uma pequena amostra desses dados para rodar o script de análise de single cell.*

In [ ]:
#script para selecionar pequena amostra a partir da matriz disponível.

O segundo passo consistirá em fazer a manipulação dos arquivos da amostra obtida e realizar uma primeira análise:
*italicized text*


In [ ]:
#Import libraries before start running the script

from typing import Literal

import anndata as ad
import lamindb as ln
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter
from scipy.sparse import csc_matrix
from scipy.special import logit
from scipy.stats import median_abs_deviation


In [ ]:
#Load sample
adata = sc.read.h5ad("nome_do_arquivo.h5ad")

If an error appear showing that genes are duplicated, we used this function to make sure each gene name is unique, by appending a number string to each duplicate.

In [ ]:
#Make names unique
adata.var_names_make_unique()
adata


**Filtering low quality cells.**

Three variates are evaluate:
1) The number of counts per barcode (count depth)
2) The number of genes per barcode
3) The fraction of counts from mitochondrial genes per barcode.

In [ ]:
#Calculate the proportions of counts for specific gene populations: mitochondrial, ribosomal, and hemoglobin genes.
#calculate the number of mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

#calculate the number of ribosomal genes (step used for additional diagnostics)
adata.var["ribo"] = adata.var_names.str.startswith("RPS", "RPL")

#calculate the number of hemoglobin genes (step used for additional diagnostics)
adata.var["hb"] = adata.var_names.str.contains(r"^HB"[ABDEGMQZ]\d*?|\w")


In [ ]:
#QC metrics calculation with scanpy
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True), percent_top=20, log1p=True)
adata

We are gonna to evaluate visually how the samples are.
To do so, we construct three plots:
1) Joint plot - evaluate the distribution of two metrics at the same time.
2) Histoplot - evaluate the distribution of a metric.
3) Scatter plot - evaluate the relation between two metrics.

In [ ]:
#Plot the QC covariates to interpret how the cells were captured.
p1 = sns.jointplot(data=adata.obs, x="log1p_total_counts", y="log1p_n_genes_by_counts", kind=hex)
p2 = sns.histoplot(adata.obs["pct_counts_mt"])
p3 = sc.pi.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")


As data becomes bigger every day, we use automatic thresholding via MAD (median absolute deviation).

Cells are marked as outliers if they differ by 5 MADs. *Important to evaluate whether this thresholds is good for your analysis*.

In [ ]:
#QC with automatic thresholding
def mad_thresholds(values: pd.Series, nmad: float) -> tuple[float, float]:
  """Thresholds 'nmads' median absolute deviations below and above the median of 'values'."""
  median, mad = np.median(values), median_abs_deviation(values)
  return median - nmad * mad, median + nmad * mad

def is_outlier(
    adata: ad.AnnData,
    metrics: str,
    nmads: float,
    side: Literal["both", "upper"] = "both",
) -> pd.Series:

    """Flag the cells beyond the MAD thresholds of 'metric', on both sides or only above."""
    values = adata.obs[metrics]
    lower, upper = mad_thresholds(values, nmads)
    return values > upper if side == "upper" else (values < lower) | (values > upper)


In [ ]:
#Logit-transform mitochondrial fraction before computing MADS
mt_fraction = (adata.obs["total_counts_mt"]+1)/(adata.obs["total_counts"] + 2)
adata.obs["logit_mt_fraction"] = logit(mt_fraction)


Metrics for mitochondrial analysis.

  Cells are flagged if exceed both: 3 MADS and a fixed cutoff of 8%.

In [ ]:
#Flagging cells
qc_filters = {
    "log1p_total_counts": (5, "both"),
    "log1p_n_genes_by_counts": (5, "both"),
    "pct_counts_in_top_20_genes": (5, "both"),
    "logit_mt_fraction": (3, "upper"),
}
flags = {
    metric: is_outlier(adata, metric, nmads, side) for metric, (nmads, side) in qc_filters.items()
}
adata.obs["outlier"] = (
    flags["log1p_total_counts"]
    | flags["log1p_n_genes_by_counts"]
    | flags["pct_counts_in_top_20_genes"]
)
adata.obs["mt_outlier"] = flags["logit_mt_fraction"] & (adata.obs["pct_counts_mt"] > 8)
adata.obs["outlier", "mt_outlier"].value_counts()

Checking the thresholds is an important step to evaluate whether the range is well defined for your data.
This step is done by ploting histograms of your different variables, such as "total_counts", "log1p_n_genes_by_counts", "pct_counts_in_top_20_genes", and "logit_mt_fraction".

In [ ]:
#Checking the thresholds
fig, axes = plt.subplots(1, len(qc_filters), figsize=(16, 3.5))
for ax, (metric, (nmads, side)) in zip(axes, qc_filters.items(), strict=True):
    sns.hitoplot(adata.obs[metric], bins=100, ax=xx)
    lower, upper = mad_thresholds(adata.obs[metric], nmads)
    for threshold in [upper] if side == "upper" else [lower, upper]:
      ax.axvline(threshold, color="red", linestyle="--")
    plt.tight_layout()
    plt.show()

After confirming that the selected thresholds are good, you filter your data, and plot the selected cell using a scatter plot.

In [ ]:
#Filtering our AnnData object based on these the thresholds
print(f"Total number of cells: {adata.n_obs}")
adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
print(f"Number of cells after filtering of low quality cells: {adata.n_obs}"]


In [ ]:
#Final graph of selected data
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
